In [1]:
import pandas as pd
df = pd.read_csv("train.csv")

#پرکردن خانه های خالی از جنس استرینگ
df["PoolQC"] = df["PoolQC"].fillna("No Pool")
df["MiscFeature"] = df["MiscFeature"].fillna("None")
df["Alley"] = df["Alley"].fillna("No alley access")
df["Fence"] = df["Fence"].fillna("No Fence")
df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["FireplaceQu"] = df["FireplaceQu"].fillna("No Fireplace")
df["GarageQual"] = df["GarageQual"].fillna("No Garage")
df["GarageFinish"] = df["GarageFinish"].fillna("No Garage")
df["GarageType"] = df["GarageType"].fillna("No Garage")
df["GarageCond"] = df["GarageCond"].fillna("No Garage")
df["BsmtFinType2"] = df["BsmtFinType2"].fillna("No Basement")
df["BsmtExposure"] = df["BsmtExposure"].fillna("No Basement")
df["BsmtCond"] = df["BsmtCond"].fillna("No Basement")
df["BsmtQual"] = df["BsmtQual"].fillna("No Basement")
df["BsmtFinType1"] = df["BsmtFinType1"].fillna("No Basement")
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

#پرکردن خانه های خالی از جنس عدد
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

df["MasVnrArea"] = df["MasVnrArea"].fillna(0)



In [2]:
#برای اینکه بفهمیم چطور ویژگی هامون رو تغییر بدیم مثلا چندتارو باهم ترکیب کنیم و بجاش یه ویژگی بسازیم 
#تقریباً همیشه این چهار سؤال را از خودت می‌پرسی:
#آیا این ویژگی برای انسان هم مهم به نظر می‌رسد؟
#آیا می‌توان از آن اطلاعات مفیدتری استخراج کرد؟
#یا مدل گفته این ویژگی مهم است؟ (Feature Importance)
#آیا در پروژه‌های مشابه این کار رایج است؟
#اگر پاسخ یکی یا چند مورد «بله» باشد، آن ویژگی کاندیدای خوبی برای Feature Engineering است.

#یه راه برای اینکه بریم سراغ فیچر ها اینه که
#ببینیم چه ویژگی هایی رو مدل مهم تشخیص داده که تو تمرین پنچم بررسی کردیم
#و در تمرین ششم هم یبار دیگه بعد اینکه مدل با لگاریتم ایگرگ ها آموزش دید ازش خواستیم ویژگی هایی که مهم دونسته رو بهمون بده که اینا رو داد
#OverallQual    0.543444
#GrLivArea      0.114076
#TotalBsmtSF    0.042573
#GarageCars     0.040925
#GarageArea     0.022275
#BsmtFinSF1     0.021560
#1stFlrSF       0.019298
#LotArea        0.014895
#YearBuilt      0.013508
#OverallCond    0.009963

#حالا یکی‌یکی بررسی کنیم که آیا ارزش Feature Engineering دارند یا نه. 
#اگر بخواهم به ترتیب اهمیت برای Feature Engineering انتخاب کنم، این‌ها را پیشنهاد می‌کنم:

#1. YearBuilt ---> HouseAge = 2010 - YearBuilt                                 #تبدیل سال ساخت به سن خانه تقریباً همیشه معنی‌دارتر است   
#2. OverallQual و GrLivArea ---> QualLivArea = OverallQual * GrLivArea         #ترکیب دو مهم‌ترین فیچر مدل.
#3. LotArea---> log1p(LotArea)                                                 #مساحت زمین معمولاً توزیع چوله دارد و لگاریتم اغلب کمک می‌کند.
#4. BsmtFinSF1  و TotalBsmtSF---> BsmtFinishedRatio = BsmtFinSF1 / TotalBsmtSF #نسبت زیرزمین تکمیل‌شده به کل زیرزمین.
#5. GarageArea و GarageCars---> GarageAreaPerCar = GarageArea / GarageCars     #میانگین فضای هر جای پارک
#6. HasBasement = (TotalBsmtSF > 0)                                            #آیا خانه زیرزمین دارد یا خیر.
#7. OverallQual و OverallCond---> QualCond = OverallQual * OverallCond         #ترکیب کیفیت و وضعیت کلی ساختمان.
#8. 1stFlrSF و GrLivArea---> FirstFloorRatio = 1stFlrSF / GrLivArea            #سهم طبقه اول از کل فضای قابل سکونت.

#اما یه نکته مهم
#من توصیه نمی‌کنم این هفت مورد را یکباره اضافه کنی.
#روش حرفه‌ای این است:
#یک Feature اضافه کن.
#Cross Validation را بگیر.
#اگر بهتر شد، نگهش دار.
#بعد برو سراغ Feature بعدی.
#به این ترتیب دقیقاً می‌فهمی کدام Feature واقعاً مدل را بهتر کرده است و کدام فقط پیچیدگی اضافه کرده است.
#این روش هم برای یادگیری عمیق‌تر بهتر است و هم در پروژه‌های واقعی قابل دفاع‌تر است.    

In [3]:
#1. ما از فیچر YearBuilt شروع میکنیم 
#در Feature Engineering یک قانون کلی وجود دارد:
#اگر یک ستون نشان‌دهنده زمان یا تاریخ باشد، معمولاً می‌توان از آن ویژگی‌های بهتری ساخت.
#مثلا سال ساخت ویژگی ای هست که میشه تبدیلش کرد به سن ساختمون
#ینی چیزی که برای خریدار مهم است این نیست که سال ساخت ۱۹۰۰ بوده یا ۲۰۰۵؛ بلکه مهم‌تر این است که:خونه 5 ساله اس یا بالای مثلا 10 سال و...
#در Feature Engineering معمولاً از ویژگی‌های زمانی، اطلاعات مفیدتری استخراج می‌کنیم.
#مثلاً از سال ساخت، تاریخ خرید، تاریخ تولد یا زمان سفارش می‌توان
#ویژگی‌هایی مثل سن، مدت زمان گذشته، فاصله زمانی، فصل، ماه یا روز هفته را ساخت.
#واسه اینکه ببینیم چه مقادیری داره این ستون از کد زیر استفاده میکنیم
df["YearBuilt"].describe()

count    1460.000000
mean     1971.267808
std        30.202904
min      1872.000000
25%      1954.000000
50%      1973.000000
75%      2000.000000
max      2010.000000
Name: YearBuilt, dtype: float64

In [4]:
#این دیتاست مربوط به فروش خانه‌ها در سال‌های ۲۰۰۶ تا ۲۰۱۰ است
#اما چیزی که واقعاً روی قیمت اثر می‌گذارد سن خانه است، نه خود سال ساخت.
#پس یک ویژگی جدید می‌سازیم.

df["HouseAge"] = 2010 - df["YearBuilt"]

#########################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8885827153326086

In [5]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8705327629654827)

In [6]:
#آخرین امتیازمیانگین np.float64(0.8698162116628433) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [7]:
#2. QualLivArea = OverallQual * GrLivArea   ترکیب کیفیت کلی خانه و متراژ فضای قابل سکونت.

df["QualLivArea"] = df["OverallQual"] * df["GrLivArea"]

########################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8874220706592878

In [8]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8754794726060062)

In [9]:
#آخرین امتیازمیانگین np.float64(0.8705327629654827) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [10]:
#3. LotArea تغییر فیچر  مساحت زمین
# معمولاً توزیع چوله دارد؛ یعنی تعداد زیادی زمین کوچک و تعداد کمی زمین بسیار بزرگ وجود دارد
#لگاریتم گرفتن باعث می‌شود اثر زمین‌های بسیار بزرگ متعادل‌تر شود و مدل راحت‌تر الگو را یاد بگیرد.
#log1p(LotArea)

df["LotArea"] = np.log1p(df["LotArea"])

#######################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.887448709829756

In [11]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8755921534180615)

In [12]:
#آخرین امتیازمیانگین np.float64(0.8754794726060062) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [13]:
#4. BsmtFinSF1  و TotalBsmtSF---> BsmtFinishedRatio = BsmtFinSF1 / TotalBsmtSF نسبت زیرزمین تکمیل‌شده به کل زیرزمین 

df["BsmtFinishedRatio"] = np.where(
    df["TotalBsmtSF"] > 0,#چون اگر TotalBsmtSF صفر باشد، از تقسیم بر صفر جلوگیری می‌شود.
    df["BsmtFinSF1"] / df["TotalBsmtSF"],
    0
)

#########################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8856559672071391

In [14]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8765079883569646)

In [15]:
#آخرین امتیازمیانگین np.float64(0.8755921534180615) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [16]:
#5.  GarageAreaPerCar = GarageArea / GarageCars  میانگین فضای هر جای پارک در گاراژ

#df["GarageAreaPerCar"] = np.where(
 #  df["GarageCars"] > 0,
 #  df["GarageArea"] / df["GarageCars"],
#    0
#)

#########################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
#X = df.drop(["Id", "SalePrice"], axis=1)
#y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
#X = pd.get_dummies(X)


#import numpy as np
#y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
#from sklearn.ensemble import RandomForestRegressor
#model = RandomForestRegressor(random_state=42)

#model.fit(X_train, y_train)

#model.score(X_test, y_test)

In [17]:
#from sklearn.model_selection import cross_val_score
#scores = cross_val_score(model,X,y,cv=5) 
#scores.mean()

In [18]:
#آخرین امتیازمیانگین np.float64(0.8765079883569646) بود 
# این فیچر باعث کاهش Cross Validation شد، بنابراین حذف می‌شود.

In [19]:
#6. OverallQual و OverallCond ---> QualCond = OverallQual * OverallCond     ترکیب کیفیت و وضعیت کلی ساختمان

df["QualCond"] = df["OverallQual"] * df["OverallCond"]

#######################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8901938705431136

In [20]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8781153516990523)

In [21]:
#آخرین امتیازمیانگین np.float64(0.8765079883569646) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [22]:
#7. HasBasement = (TotalBsmtSF > 0)  آیا خانه زیرزمین دارد یا خیر.
#ایده این است که شاید داشتن یا نداشتن زیرزمین خودش روی قیمت اثر بگذارد، حتی اگر مساحت زیرزمین را جداگانه هم داشته باشیم.

#df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)


######################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
#X = df.drop(["Id", "SalePrice"], axis=1)
#y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
#X = pd.get_dummies(X)


#import numpy as np
#y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
#from sklearn.ensemble import RandomForestRegressor
#model = RandomForestRegressor(random_state=42)

#model.fit(X_train, y_train)

#model.score(X_test, y_test)

In [23]:
#from sklearn.model_selection import cross_val_score
#scores = cross_val_score(model,X,y,cv=5) 
#scores.mean()

In [24]:
#آخرین امتیازمیانگین np.float64(0.8781153516990523) بود 
# این فیچر باعث کاهش Cross Validation شد، بنابراین حذف می‌شود.

In [25]:
#8. 1stFlrSF و GrLivArea---> FirstFloorRatio = 1stFlrSF / GrLivArea   نسبت متراژ طبقه اول به کل فضای قابل سکونت خانه


#df["FirstFloorRatio"] = np.where(
  #  df["GrLivArea"] > 0,
  #  df["1stFlrSF"] / df["GrLivArea"],
  # 0
#)
####################

#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
#X = df.drop(["Id", "SalePrice"], axis=1)
#y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
#X = pd.get_dummies(X)


#import numpy as np
#y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
#from sklearn.ensemble import RandomForestRegressor
#model = RandomForestRegressor(random_state=42)

#model.fit(X_train, y_train)

#model.score(X_test, y_test)

In [26]:
#from sklearn.model_selection import cross_val_score
#scores = cross_val_score(model,X,y,cv=5) 
#scores.mean()

In [27]:
#آخرین امتیازمیانگین np.float64(0.8781153516990523) بود 
# این فیچر باعث کاهش Cross Validation شد، بنابراین حذف می‌شود.

In [28]:
#********************************************************تا اینجا فیچرهایی که برای مدل مهم بودن رو اومدیم روشون حرکت زدیم
#از اینجا به بعد مهندسی فیچر رو مبتنی بردانش مسیله میبریم جلو نه فقط فیچرهای مهم

#حالا منطقی‌ترین مرحله بعدی این است:
#Feature Engineering بر اساس دانش قیمت‌گذاری خانه (Domain Knowledge)
#این یعنی به جای اینکه فقط به Feature Importance نگاه کنیم، از خودمان بپرسیم:
#«اگر من خریدار خانه بودم، چه چیزهایی را به صورت ترکیبی بررسی می‌کردم؟»
#من این ترتیب را پیشنهاد می‌کنم:
#اولویت	Feature	                  دلیل
#🥇 1	TotalBath    	تعداد کل سرویس‌های بهداشتی معمولاً از ستون‌های جداگانه مربوط به فیچر سرویس بهداشتی معنادارتر است 
#🥈 2	TotalPorchSF	مجموع همه ایوان‌ها و فضاهای بیرونی
#🥉 3	TotalOutdoorSF	ترکیب ایوان، دک و سایر فضاهای بیرونی
#   4	HasFireplace    داشتن شومینه یا نداشتن آن
#   5	HasPool	         داشتن استخر یا نداشتن آن
#   6   HasFence	    داشتن حصار یا نداشتن آن
#   7	HasGarage    	دوباره امتحان می‌کنیم، ولی این بار در کنار Featureهای جدید دیگر

In [29]:
#چرا از TotalBath شروع کنیم؟
#چون از نظر مهندسی ویژگی یکی از معروف‌ترین Featureها در پروژه‌های قیمت مسکن است.

#  TotalBath مجموع تعداد سرویس‌های بهداشتی  
#الان مدل این ستون‌ها را جداگانه دارد:
#FullBath
#HalfBath
#BsmtFullBath
#BsmtHalfBath
#اما خریدار معمولاً به تعداد کل سرویس‌های بهداشتی اهمیت می‌دهد.
#نیم‌سرویس هم ارزش کمتری نسبت به سرویس کامل دارد، پس آن را ۰.۵ حساب می‌کنیم.

df["TotalBath"] = (
    df["FullBath"]
    + 0.5 * df["HalfBath"]
    + df["BsmtFullBath"]
    + 0.5 * df["BsmtHalfBath"]
)

#######################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8913733776216013

In [30]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8787751894851817)

In [31]:
#آخرین امتیازمیانگین np.float64(0.8781153516990523) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [32]:
#TotalPorchSF
#در دیتاست، ایوان‌ها و فضاهای بیرونی در چند ستون جداگانه ذخیره شده‌اند: OpenPorchSF, EnclosedPorch, 3SsnPorch,  ScreenPorch
#برای خریدار معمولاً مهم است که در مجموع چقدر فضای ایوان دارد، نه اینکه این فضا در چند ستون مختلف تقسیم شده باشد.

#df["TotalPorchSF"] = ( # مجموع مساحت تمام انواع ایوان و فضاهای بیرونی خانه
#    df["OpenPorchSF"]
 #  + df["EnclosedPorch"]
 #  + df["3SsnPorch"]
#    + df["ScreenPorch"]
#)
####################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
#X = df.drop(["Id", "SalePrice"], axis=1)
#y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
#X = pd.get_dummies(X)


#import numpy as np
#y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
#from sklearn.ensemble import RandomForestRegressor
#model = RandomForestRegressor(random_state=42)

#model.fit(X_train, y_train)

#model.score(X_test, y_test)

In [33]:
#from sklearn.model_selection import cross_val_score
#scores = cross_val_score(model,X,y,cv=5) 
#scores.mean()

In [34]:
#آخرین امتیازمیانگین np.float64(0.8787751894851817) بود 
# این فیچر باعث کاهش Cross Validation شد، بنابراین حذف می‌شود.

In [35]:
##TotalOutdoorSF مجموع مساحت تمام فضاهای بیرونی خانه (دک و انواع ایوان)
#برویم. این Feature از نظر من منطقی‌تر از TotalPorchSF است،
#چون خریدار معمولاً به کل فضای قابل استفاده بیرونی اهمیت می‌دهد، نه فقط ایوان.

#df["TotalOutdoorSF"] = (
#    df["WoodDeckSF"]
 #  + df["OpenPorchSF"]
  #  + df["EnclosedPorch"]
  # + df["3SsnPorch"]
  # + df["ScreenPorch"]
#)

#######################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
#X = df.drop(["Id", "SalePrice"], axis=1)
#y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
#X = pd.get_dummies(X)


#import numpy as np
#y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
#from sklearn.ensemble import RandomForestRegressor
#model = RandomForestRegressor(random_state=42)

#model.fit(X_train, y_train)

#model.score(X_test, y_test)

In [36]:
#from sklearn.model_selection import cross_val_score
#scores = cross_val_score(model,X,y,cv=5) 
#scores.mean()

In [37]:
#آخرین امتیازمیانگین np.float64(0.8787751894851817) بود 
# این فیچر باعث کاهش Cross Validation شد، بنابراین حذف می‌شود.

In [38]:
#  TotalSF = TotalBsmtSF+ GrLivArea # مجموع فضای قابل سکونت و زیرزمین برای ساخت متراژ کل ساختمان
# یکی از معروف‌ترین فیچرهای مسابقه قیمت مسکن کگل
#چون خریدار معمولاً به کل فضای خانه اهمیت می‌دهد، نه اینکه زیرزمین و طبقات بالا جداگانه چند متر باشند.
#در بسیاری از راه‌حل‌های موفق Kaggle، ویژگی TotalSF یکی از ویژگی‌های مؤثر بوده است.

df["TotalSF"] = df["TotalBsmtSF"] + df["GrLivArea"]

#########################

#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8889363498169265

In [39]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8823234063859549)

In [40]:
#آخرین امتیازمیانگین np.float64(0.8787751894851817) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [41]:
# YearRemodAdd
#یکی از معروف‌ترین ویژگی‌های این دیتاست: سن بازسازی خانه
#الان فقط سال بازسازی را داریم: YearRemodAdd
#اما چیزی که برای مدل قابل فهم‌تر است: چند سال از آخرین بازسازی گذشته است؟
##سن آخرین بازسازی خانه

df["RemodAge"] = 2010 - df["YearRemodAdd"]

####################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8906523233824137

In [42]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8827673180362658)

In [43]:
#آخرین امتیازمیانگین np.float64(0.8823234063859549) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [44]:
# آیا خانه حداقل یک شومینه دارد یا خیر
df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
####################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8905802447615209

In [45]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8829414652402301)

In [46]:
#آخرین امتیازمیانگین np.float64(0.8827673180362658) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [47]:
# آیا خانه استخر دارد یا خیر
#df["HasPool"] = (df["PoolArea"] > 0).astype(int)

#####################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
#X = df.drop(["Id", "SalePrice"], axis=1)
#y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
#X = pd.get_dummies(X)


#import numpy as np
#y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
#from sklearn.ensemble import RandomForestRegressor
#model = RandomForestRegressor(random_state=42)

#model.fit(X_train, y_train)

#model.score(X_test, y_test)

In [48]:
#from sklearn.model_selection import cross_val_score
#scores = cross_val_score(model,X,y,cv=5) 
#scores.mean()

In [49]:
#آخرین امتیازمیانگین np.float64(0.8829414652402301) بود 
# این فیچر باعث کاهش Cross Validation شد، بنابراین حذف می‌شود.

In [50]:
# آیا خانه حصار دارد یا خیر
df["HasFence"] = (df["Fence"] != "No Fence").astype(int)
####################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8904020277530056

In [51]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.883097620801823)

In [52]:
#آخرین امتیازمیانگین np.float64(0.8829414652402301) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [53]:
#HasGarage  # آیا خانه گاراژ دارد یا خیر
#یکی از بهترین Featureهایی که تقریباً در اکثر راه‌حل‌های خوب Kaggle استفاده می‌شود:
#آیا خانه گاراژ دارد یا نه؟
#الان مدل ستون‌های زیادی درباره گاراژ دارد:
#GarageArea
#GarageCars
#GarageType
#GarageQual
#...
#اما گاهی داشتن یا نداشتن گاراژ خودش یک اطلاعات مهم است.
#df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
#اگر GarageArea بزرگ‌تر از صفر باشد ⇒ مقدار ۱
#اگر GarageArea صفر باشد ⇒ مقدار ۰
#########################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
#X = df.drop(["Id", "SalePrice"], axis=1)
#y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
#X = pd.get_dummies(X)


#import numpy as np
#y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
#from sklearn.ensemble import RandomForestRegressor
#model = RandomForestRegressor(random_state=42)

#model.fit(X_train, y_train)

#model.score(X_test, y_test)

In [54]:
#from sklearn.model_selection import cross_val_score
#scores = cross_val_score(model,X,y,cv=5) 
#scores.mean()

In [55]:
#آخرین امتیازمیانگین np.float64(0.883097620801823) بود 
# این فیچر باعث کاهش Cross Validation شد، بنابراین حذف می‌شود.

In [56]:
df["QualFireplace"] = df["OverallQual"] * df["Fireplaces"]
####################
#بعدش میریم ایکس و ایگرگ تعیین میکنیم  تا مدل رو آموزش بدیم و اسکور بگیریم 

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)


#import numpy as np
y = np.log1p(y)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

model.fit(X_train, y_train)

model.score(X_test, y_test)

0.8893477416992075

In [57]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) 
scores.mean()

np.float64(0.8834220102732209)

In [58]:
#آخرین امتیازمیانگین np.float64(0.883097620801823) بود 
# این فیچر باعث بهبود Cross Validation شد، بنابراین نگه داشته می‌شود.

In [59]:
#تو این Featureها را امتحان کردی:

#❌ GarageAreaPerCar
#❌ HasBasement
#❌ FirstFloorRatio
#❌ TotalPorchSF
#❌ TotalOutdoorSF
#❌ HasPool
#❌ HasGarage
#و هیچ‌کدام باعث بهبود Cross Validation نشدند.

#و این‌ها را نگه داشتی:

#✅ HouseAge
#✅ QualLivArea
#✅ QualFireplace
#✅ log1p(LotArea)
#✅ BsmtFinishedRatio
#✅ QualCond
#✅ TotalBath
#✅ TotalSF
#✅ RemodAge
#✅ HasFireplace
#✅ HasFence

In [60]:
# حالا که یه سری فیچر جدید تعریف کردم میخوام ببینم الان مدلم چه ویژگی هایی رو مهم میدونه

importance = pd.Series( 
    model.feature_importances_,  
   index=X.columns 
)
pd.set_option("display.float_format", "{:.6f}".format)#برای اینکه عددهارو با نماد علمی نشون نده و اعشاری معمولی بهم بده
pd.set_option("display.max_rows", None)
importance.sort_values(ascending=False).head(30)

QualLivArea         0.600108
TotalSF             0.107287
QualFireplace       0.023898
TotalBsmtSF         0.022253
GarageCars          0.017661
OverallQual         0.013929
QualCond            0.013328
HouseAge            0.012651
YearBuilt           0.011976
BsmtFinSF1          0.010930
TotalBath           0.010866
LotArea             0.009753
GarageArea          0.009389
1stFlrSF            0.007242
GarageYrBlt         0.006163
YearRemodAdd        0.005296
MSZoning_RM         0.005003
BsmtUnfSF           0.004994
RemodAge            0.004860
LotFrontage         0.004383
OverallCond         0.004023
2ndFlrSF            0.003756
CentralAir_N        0.003553
BsmtQual_Gd         0.003319
GrLivArea           0.003317
BsmtFinishedRatio   0.003223
OpenPorchSF         0.003142
MoSold              0.002920
BsmtQual_Ex         0.002902
CentralAir_Y        0.002500
dtype: float64

In [61]:
#۱. QualLivArea فوق‌العاده عمل کرده است.
#این یعنی ترکیبی که ساختی از خود OverallQual و GrLivArea خیلی اطلاعات بیشتری به مدل داده است.
################################################
#۲. TotalSF هم موفق بوده.
#حتی از TotalBsmtSF، 1stFlrSF و GrLivArea به تنهایی مهم‌تر شده است.
################################################
#۳. بعضی فیچرهای مهندسی‌شده موفق بوده‌اند ولی اطلاعاتشان را با فیچرهای اصلی به اشتراک گذاشته‌اند.
#HouseAge             0.0106
#YearBuilt            0.0124

#RemodAge             0.0050
#YearRemodAdd         0.0044
#این طبیعی است. چون HouseAge از YearBuilt ساخته شده و RandomForest اهمیت را بینشان تقسیم می‌کند.
################################################
#۴. بعضی فیچرهای مهندسی‌شده تقریباً بی‌اثر شده‌اند.
#HasFireplace         0.000565
#HasFence             0.000174
#یعنی وجودشان ضرری نزده، ولی تأثیر زیادی هم نداشته است.
################################################
#چیزی که چشم من را گرفت
#OpenPorchSF      0.0033
#WoodDeckSF       0.0021
#EnclosedPorch    0.0010
#ScreenPorch      0.0004
#یادمان هست که TotalPorchSF را ساختیم و نتیجه نگرفت.
#پس دیگر لازم نیست دوباره سراغ Featureهای مربوط به Porch برویم.
#همچنین 
#GarageCars    0.019
#GarageArea    0.008
#ولی
#GarageAreaPerCar را قبلاً امتحان کردی و جواب نگرفتی؛ پس آن هم بسته شده است.

In [62]:
#الان اگر بخواهم ادامه بدهم، دو مسیر منطقی داریم:
#1. Interaction Feature Engineering :یعنی ساخت فیچر از ترکیب دو فیچر مهم دیگر، مثل نسبت‌ها یا حاصل‌ضرب‌های هدفمند
#2. Hyperparameter Tuning که معمولاً در این مرحله بازده بیشتری از ساخت Featureهای جدید دارد.